# Formulario base de inscripción · Taller «Fiestas en equilibrio»

Este cuaderno crea en **tu Google Drive** el formulario de Google del kit del curso ProPatiens (Asociación Equilibrio, **ficticia**): preguntas, cláusula de protección de datos, autorizaciones de imagen y comunicaciones, y la cabecera con el logo y los colores de la asociación.

Al final te da tres enlaces:

- **Enlace para participantes («Hacer una copia»)**: quien lo abre tiene su propio formulario en su Drive con un clic.
- **Enlace para responder**: el que se pega en DifundIA.
- **Enlace para editar**.

**Cómo se usa:** menú **Entorno de ejecución → Ejecutar todas** y acepta el permiso de Google cuando lo pida. Tarda menos de un minuto.

Google no deja cambiar el color del tema desde aquí: si quieres el verde de la asociación, en el formulario pulsa el icono de la paleta → **Color** → `#1F7A5C` (las copias lo heredan).

## 1 · Permiso para usar tu Google Drive

In [ ]:
from google.colab import auth
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
forms = build('forms', 'v1', credentials=creds)
drive = build('drive', 'v3', credentials=creds)
print('✓ Conectado con tu cuenta de Google')

## 2 · Datos del formulario

Para otra asociación u otra actividad, cambia solo esta celda.

In [ ]:
ASOCIACION = 'Asociación Equilibrio de Personas con Diabetes de Valdeclara'
DIRECCION  = 'calle de los Almendros 14, Valdeclara'
CORREO     = 'info@asociacion-equilibrio.example'

TITULO = 'Inscripción · Taller de cocina «Fiestas en equilibrio»'
DESCRIPCION = (
    'Tres recetas para disfrutar de las fiestas en familia, pensado para personas con diabetes y quien cocina en casa.\n\n'
    'Cuándo: sábado, 12 de diciembre de 2026, de 10:30 a 13:30.\n'
    'Dónde: cocina del Centro Social de Valdeclara, calle Mayor 22.\n'
    'Imparte: Nuria Calvo, enfermera educadora en diabetes, con el grupo de cocina de la asociación.\n'
    'Plazas: 20, por orden de inscripción. Inscripción abierta hasta el jueves 10 de diciembre.\n'
    'Precio: gratuito para socios; 5 € para personas no socias.\n\n'
    'Te confirmaremos la plaza por teléfono o correo antes del 10 de diciembre. '
    f'Si tienes alguna alergia o intolerancia alimentaria, cuéntanoslo en la sede o escribiendo a {CORREO}.'
)

# Cabecera con el logo y los colores (1600 × 400 px). Tiene que ser una dirección pública de internet.
CABECERA_URL = 'https://fborrasumh.github.io/propatiens/assets/cabecera_formulario_equilibrio.png'

PROTECCION_DATOS = (
    f'Responsable: {ASOCIACION}, {DIRECCION}, {CORREO}.\n\n'
    'Finalidad: gestionar tu inscripción en el taller, asignar las plazas y avisarte de cualquier cambio. '
    'Solo si lo autorizas más abajo: publicar fotos o vídeos del taller y enviarte información de próximas actividades.\n\n'
    f'Legitimación: tu consentimiento, que puedes retirar cuando quieras escribiendo a {CORREO}.\n\n'
    'Destinatarios: no cedemos tus datos a nadie, salvo obligación legal. Se recogen y guardan con Google Forms; '
    'Google actúa como proveedor del servicio y puede tratarlos fuera de la Unión Europea con las garantías que exige el RGPD.\n\n'
    'Conservación: los datos de inscripción, hasta un año después del taller; las fotos y vídeos, mientras no retires tu autorización.\n\n'
    'Derechos: puedes acceder a tus datos, rectificarlos, suprimirlos, oponerte a su tratamiento, limitarlo y pedir su portabilidad '
    f'escribiendo a {CORREO}. Si crees que no los tratamos bien, puedes reclamar ante la Agencia Española de Protección de Datos (www.aepd.es).\n\n'
    'Por favor, no escribas en este formulario nada sobre tu salud.'
)
print('✓ Datos preparados')

## 3 · Crear el formulario con sus preguntas

In [ ]:
def texto(titulo, ayuda='', obligatoria=True):
    return {'title': titulo, 'description': ayuda,
            'questionItem': {'question': {'required': obligatoria, 'textQuestion': {}}}}

def opciones(titulo, valores, tipo='RADIO', ayuda='', obligatoria=True):
    return {'title': titulo, 'description': ayuda,
            'questionItem': {'question': {'required': obligatoria,
                                          'choiceQuestion': {'type': tipo, 'options': [{'value': v} for v in valores]}}}}

ELEMENTOS = [
    texto('Nombre y apellidos'),
    texto('Teléfono', 'Te llamaremos o escribiremos por WhatsApp solo para confirmar tu plaza o avisarte de cambios.'),
    texto('Correo electrónico', 'Opcional.', obligatoria=False),
    opciones('¿Eres socio o socia de la asociación?', ['Sí (gratuito)', 'No (5 €, se pagan el mismo día)']),
    opciones('¿Cuántas plazas reservas, contándote a ti?', ['1', '2', '3'], tipo='DROP_DOWN',
             ayuda='Si viene alguna persona menor de edad, debe venir acompañada de una persona adulta, que es quien la inscribe.'),
    {'title': 'Protección de datos y autorizaciones', 'pageBreakItem': {}},
    {'title': 'Información básica sobre protección de datos', 'description': PROTECCION_DATOS, 'textItem': {}},
    opciones('Consentimiento para la inscripción',
             ['He leído la información sobre protección de datos y acepto que la asociación trate mis datos para gestionar mi inscripción.'],
             tipo='CHECKBOX'),
    opciones('Fotos y vídeos del taller',
             ['Sí, autorizo que la asociación publique fotos o vídeos del taller en los que aparezca, en su web, su boletín y sus redes sociales.',
              'No autorizo la publicación de mi imagen.'],
             ayuda='Si inscribes a menores, esta respuesta vale también para ellos y la das como madre, padre o tutor.'),
    opciones('Información de próximas actividades',
             ['Sí, quiero recibir información de las actividades de la asociación por correo o WhatsApp.', 'No, gracias.']),
]

try:
    form = forms.forms().create(body={'info': {'title': TITULO, 'documentTitle': TITULO}}).execute()
except HttpError as e:
    raise SystemExit('✗ Google no deja crear el formulario desde Colab con esta cuenta:\n' + str(e) +
                     '\nAlternativa: el script crear_formulario_inscripcion.gs del kit (script.google.com).')
FORM_ID = form['formId']

peticiones = [{'updateFormInfo': {'info': {'title': TITULO, 'description': DESCRIPCION}, 'updateMask': 'title,description'}}]
peticiones += [{'createItem': {'item': el, 'location': {'index': i}}} for i, el in enumerate(ELEMENTOS)]
forms.forms().batchUpdate(formId=FORM_ID, body={'requests': peticiones}).execute()

# la cabecera va arriba del todo; si no se puede descargar la imagen, el formulario sigue igual de válido
try:
    forms.forms().batchUpdate(formId=FORM_ID, body={'requests': [{'createItem': {
        'item': {'imageItem': {'image': {'sourceUri': CABECERA_URL, 'properties': {'alignment': 'CENTER', 'width': 740}}}},
        'location': {'index': 0}}}]}).execute()
    print('✓ Cabecera añadida')
except HttpError as e:
    print('⚠ No se pudo añadir la cabecera (¿la dirección de la imagen es pública?). El resto del formulario está bien.')

print(f'✓ Formulario creado con {len(ELEMENTOS)} elementos')

## 4 · Publicarlo y obtener los enlaces

In [ ]:
# publicar y aceptar respuestas (los formularios antiguos ya nacen publicados)
try:
    forms.forms().setPublishSettings(formId=FORM_ID, body={
        'publishSettings': {'publishState': {'isPublished': True, 'isAcceptingResponses': True}}}).execute()
except Exception:
    pass

# «cualquiera con el enlace puede ver»: es lo que necesita el enlace de copia; ver no permite editar el original
drive.permissions().create(fileId=FORM_ID, body={'type': 'anyone', 'role': 'reader'}).execute()

info = forms.forms().get(formId=FORM_ID).execute()
COPIA = f'https://docs.google.com/forms/d/{FORM_ID}/copy'
RESPONDER = info['responderUri']
EDITAR = f'https://docs.google.com/forms/d/{FORM_ID}/edit'

from IPython.display import Markdown, display
display(Markdown(f"""### ✓ Listo

| Para qué | Enlace |
|---|---|
| **Participantes: hacer una copia en su Drive** | {COPIA} |
| Responder (pegar en DifundIA) | {RESPONDER} |
| Editar el formulario | {EDITAR} |

Las respuestas se ven en la pestaña **Respuestas** del formulario. Para tenerlas en una hoja de cálculo: **Respuestas → Vincular con Hojas de cálculo**.
"""))